In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

#### Summarization Middleware

It helps to automatically summarize conversation hisyory when approaching token limit or other flags that we given, preserve recent messages while compressing older context. 

In [5]:
#### Summarization Middleware

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

## from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## Messagebase summarization

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", max_tokens=100)

summarization = SummarizationMiddleware(
    model = llm,
    trigger = ("tokens", 500),
    keep = ("tokens", 200)
)

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[summarization],
    checkpointer=InMemorySaver()
)

In [6]:
config = {"configurable":{"thread_id":"test-1"}}

In [7]:
## test data

questions = [
    "What is 2+2?",
    "What is the capital of France?",
    "What is the square root of 16?",
    "What is the cube root of 27?",
    "what is 10*2?",
    "What is 246/4?",
    "What is 35-7?",
    "What is 12+12?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='c527badf-cecd-4b77-9926-4eef24d16efc'), AIMessage(content='2 + 2 is 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.011036812, 'completion_tokens_details': None, 'prompt_time': 0.001995539, 'prompt_tokens_details': None, 'queue_time': 0.045857211, 'total_time': 0.013032351}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cdd4c-0ba3-7ce1-8e4d-73e962ab12ba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='c527badf-cecd-4b77-9926-4eef24d16efc'), AIMessag